In [1]:
!pip install faker

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import pandas as pd
import numpy as np
import networkx as nx
from faker import Faker
import random
from tqdm import tqdm

fake = Faker()
np.random.seed(42)
random.seed(42)

NUM_ACCOUNTS = 5000
NUM_MULES = 200

In [3]:
devices = []

device_ids = [f"D{i}" for i in range(4000)]

# Normal accounts
for acc in range(NUM_ACCOUNTS - NUM_MULES):
    devices.append({
        "device_id": random.choice(device_ids),
        "account_id": f"ACC{acc+1:05d}",
        "device_type": random.choice(["Mobile", "Laptop"]),
        "device_os": random.choice(["Android", "iOS", "Windows"]),
        "device_model": fake.word(),
        "vpn_usage_flag": False,
        "device_trust_score": np.random.uniform(0.7, 1.0)
    })

# Mule accounts share devices
shared_devices = random.sample(device_ids, 50)

for acc in range(NUM_ACCOUNTS - NUM_MULES, NUM_ACCOUNTS):
    devices.append({
        "device_id": random.choice(shared_devices),
        "account_id": f"ACC{acc+1:05d}",
        "device_type": "Mobile",
        "device_os": "Android",
        "device_model": fake.word(),
        "vpn_usage_flag": True,
        "device_trust_score": np.random.uniform(0.1, 0.5)
    })

devices_df = pd.DataFrame(devices)
devices_df.to_csv("devices.csv", index=False)

In [4]:
transactions = []
transaction_id = 0

for acc in range(NUM_ACCOUNTS - NUM_MULES):
    num_tx = np.random.randint(20, 60)

    for _ in range(num_tx):
        receiver = random.randint(0, NUM_ACCOUNTS - NUM_MULES)

        transactions.append({
            "transaction_id": f"T{transaction_id}",
            "sender_account_id": f"ACC{acc+1:05d}",
            "receiver_account_id": f"ACC{receiver+1:05d}",
            "amount": round(np.random.uniform(100, 10000), 2),
            "channel": random.choice(["Mobile", "UPI", "Web", "ATM"]),
            "timestamp": fake.date_time_this_year(),
            "is_fraud": 0,
            "fraud_type": "None"
        })
        transaction_id += 1

In [5]:
mule_accounts = [f"ACC{i+1:05d}" for i in range(NUM_ACCOUNTS - NUM_MULES, NUM_ACCOUNTS)]
for mule in mule_accounts[:50]:
    victim = random.choice([f"ACC{i+1:05d}" for i in range(2000)])

    for _ in range(5):
        transactions.append({
            "transaction_id": f"T{transaction_id}",
            "sender_account_id": victim,
            "receiver_account_id": mule,
            "amount": round(np.random.uniform(20000, 80000), 2),
            "channel": random.choice(["Mobile", "UPI"]),
            "timestamp": fake.date_time_this_year(),
            "is_fraud": 1,
            "fraud_type": "Star_Mule"
        })
        transaction_id += 1
ring = mule_accounts[50:100]

for i in range(len(ring)):
    transactions.append({
        "transaction_id": f"T{transaction_id}",
        "sender_account_id": ring[i],
        "receiver_account_id": ring[(i+1) % len(ring)],
        "amount": round(np.random.uniform(30000, 60000), 2),
        "channel": "UPI",
        "timestamp": fake.date_time_this_year(),
        "is_fraud": 1,
        "fraud_type": "Ring_Mule"
    })
    transaction_id += 1
chain = mule_accounts[100:150]

for i in range(len(chain) - 1):
    transactions.append({
        "transaction_id": f"T{transaction_id}",
        "sender_account_id": chain[i],
        "receiver_account_id": chain[i+1],
        "amount": round(np.random.uniform(50000, 90000), 2),
        "channel": "Mobile",
        "timestamp": fake.date_time_this_year(),
        "is_fraud": 1,
        "fraud_type": "Chain_Mule"
    })
    transaction_id += 1
for mule in mule_accounts[150:]:
    victim = random.choice([f"ACC{i+1:05d}" for i in range(2000)])

    channels = ["Mobile", "UPI", "ATM"]

    for ch in channels:
        transactions.append({
            "transaction_id": f"T{transaction_id}",
            "sender_account_id": victim,
            "receiver_account_id": mule,
            "amount": round(np.random.uniform(40000, 70000), 2),
            "channel": ch,
            "timestamp": fake.date_time_this_year(),
            "is_fraud": 1,
            "fraud_type": "Cross_Channel"
        })
        transaction_id += 1
transactions_df = pd.DataFrame(transactions)
transactions_df.to_csv("transactions.csv", index=False)

In [6]:
logins = []

for acc in range(NUM_ACCOUNTS):
    num_logins = np.random.randint(10, 50)

    for _ in range(num_logins):
        logins.append({
            "account_id": f"ACC{acc+1:05d}",
            "ip_address": fake.ipv4(),
            "country": fake.country(),
            "session_duration": np.random.randint(1, 60),
            "timestamp": fake.date_time_this_year()
        })

logins_df = pd.DataFrame(logins)
logins_df.to_csv("logins.csv", index=False)

In [7]:
G = nx.DiGraph()

for _, row in transactions_df.iterrows():
    G.add_edge(row["sender_account_id"], row["receiver_account_id"])
features = []

pagerank = nx.pagerank(G)
between = nx.betweenness_centrality(G)

for node in G.nodes():
    features.append({
        "account_id": node,
        "in_degree": G.in_degree(node),
        "out_degree": G.out_degree(node),
        "betweenness": between[node],
        "pagerank": pagerank[node]
    })

network_df = pd.DataFrame(features)
network_df.to_csv("network_features.csv", index=False)

CONVERTING TO NODES AND EDGES\

In [8]:
import pandas as pd

customers = pd.read_csv("customers.csv")
devices = pd.read_csv("devices.csv")
transactions = pd.read_csv("transactions.csv")
network = pd.read_csv("network_features.csv")

C:\Users\JHANANISHRI\AppData\Local\Temp\ipykernel_7568\519979824.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv("transactions.csv")


In [9]:
accounts = customers["account_id"].unique()
account_mapping = {acc: i for i, acc in enumerate(accounts)}
features = customers.merge(network, on="account_id", how="left")
features["customer_risk_category"] = features["customer_risk_category"].map({
    "low": 0,
    "medium": 1,
    "high": 2
})
feature_cols = [
    "credit_score",
    "average_monthly_balance",
    "customer_risk_category",
    "in_degree",
    "out_degree",
    "betweenness",
    "pagerank"
]

In [10]:
import torch

X = torch.tensor(features[feature_cols].fillna(0).values, dtype=torch.float)

In [11]:
edges = []

for _, row in transactions.iterrows():
    s = account_mapping.get(row["sender_account_id"])
    r = account_mapping.get(row["receiver_account_id"])

    if s is not None and r is not None:
        edges.append([s, r])

In [12]:
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [13]:
fraud_accounts = set(
    transactions[transactions["is_fraud"] == 1]["receiver_account_id"]
)
labels = []

for acc in accounts:
    labels.append(1 if acc in fraud_accounts else 0)

y = torch.tensor(labels, dtype=torch.long)

In [14]:
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html

'pip' is not recognized as an internal or external command,
operable program or batch file.


'pip' is not recognized as an internal or external command,
operable program or batch file.


In [15]:
from torch_geometric.data import Data

graph_data = Data(x=X, edge_index=edge_index, y=y)

c:\Users\JHANANISHRI\anaconda3\envs\diffusion-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1️⃣ Data Splitting

We perform a **Stratified train/validation/test split (70/15/15)**. 
* **Why Stratified Split?** Since fraud datasets are heavily imbalanced (very few fraud nodes compared to normal ones), a random split might result in train/val/test sets with no fraud cases at all. Stratification ensures that the ratio of fraud to normal nodes is identical across all splits.

In [16]:
import torch
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

labels = graph_data.y.numpy()
indices = np.arange(len(labels))

# First split: 70% Train, 30% Temp (Val + Test)
train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, labels, test_size=0.30, stratify=labels, random_state=42
)

# Second split: 50% Val, 50% Test of the remaining 30% (yielding 15% / 15% of total)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Convert to boolean masks for PyTorch Geometric
train_mask = torch.zeros(len(labels), dtype=torch.bool)
val_mask = torch.zeros(len(labels), dtype=torch.bool)
test_mask = torch.zeros(len(labels), dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

graph_data.train_mask = train_mask
graph_data.val_mask = val_mask
graph_data.test_mask = test_mask

print(f"Train nodes: {train_mask.sum().item()}")
print(f"Val nodes: {val_mask.sum().item()}")
print(f"Test nodes: {test_mask.sum().item()}")


Train nodes: 3500
Val nodes: 750
Test nodes: 750


### 2️⃣ Model Architecture (GraphSAGE)

We implement a 3-layer GraphSAGE model with hidden dimensions of 64 and 32, complete with Dropout and ReLU activations.

In [17]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class FraudGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels_1, hidden_channels_2, out_channels, dropout=0.5):
        super(FraudGraphSAGE, self).__init__()
        self.dropout = dropout
        
        # 1st GraphSAGE Layer (Input -> 64)
        self.conv1 = SAGEConv(in_channels, hidden_channels_1)
        # 2nd GraphSAGE Layer (64 -> 32)
        self.conv2 = SAGEConv(hidden_channels_1, hidden_channels_2)
        # 3rd Output Layer (32 -> 2 classes)
        self.conv3 = SAGEConv(hidden_channels_2, out_channels)

    def forward(self, x, edge_index):
        # Layer 1
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Layer 2
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Layer 3 (Output)
        x = self.conv3(x, edge_index)
        
        return x # Return raw logits for CrossEntropyLoss


### 3️⃣ Training Setup

* **Class Weights:** We compute inverse class weights because there are vastly more normal transactions than fraudulent ones. The CrossEntropyLoss uses these weights to heavily penalize the model when it misclassifies a rare fraud node, forcing the model to pay attention to minority classes.
* **Early Stopping:** If the model's performance on the validation set doesn't improve for a set number of epochs (patience=20), training halts. This prevents overfitting onto the training data and saves computational time.

In [18]:
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Push data to device
graph_data = graph_data.to(device)

# Compute class weights based on the training set
train_labels = graph_data.y[graph_data.train_mask].cpu().numpy()
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Computed Class Weights: {class_weights_tensor}")

# Initialize Model, Optimizer, Loss Function
model = FraudGraphSAGE(in_channels=graph_data.num_features, 
                       hidden_channels_1=64, 
                       hidden_channels_2=32, 
                       out_channels=2).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# Setup Early Stopping parameters
max_epochs = 200
patience = 20
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_path = 'best_gnn_model.pt'


Using device: cuda


Computed Class Weights: tensor([ 0.5207, 12.5899], device='cuda:0')


### 4️⃣ Training Loop

In [19]:
for epoch in range(1, max_epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    out = model(graph_data.x, graph_data.edge_index)
    
    # Compute loss on training nodes only
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    
    # Backward pass and optimize
    loss.backward()
    optimizer.step()
    
    # Validation phase
    model.eval()
    with torch.no_grad():
        val_out = model(graph_data.x, graph_data.edge_index)
        val_loss = criterion(val_out[graph_data.val_mask], graph_data.y[graph_data.val_mask])
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f}')
    
    # Early Stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path) # Save best model
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}! Best Val Loss: {best_val_loss:.4f}")
            break


Epoch: 010, Train Loss: 1892.6761, Val Loss: 0.6255
Epoch: 020, Train Loss: 33.5068, Val Loss: 0.6934
Epoch: 030, Train Loss: 8.4516, Val Loss: 0.6950

Early stopping triggered at epoch 30! Best Val Loss: 0.6255


### 5️⃣ & 6️⃣ Evaluation & Model Loading

* **Why ROC-AUC?** In imbalanced datasets, a model could achieve 99% accuracy simply by predicting "normal" every time. ROC-AUC (Area Under the Receiver Operating Characteristic Curve) measures the model's ability to distinguish between classes across all possible classification thresholds, making it a much more reliable metric for fraud detection.

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Load the best model back for inference
model.load_state_dict(torch.load(best_model_path))
model.eval()

with torch.no_grad():
    out = model(graph_data.x, graph_data.edge_index)
    probs = F.softmax(out, dim=1)
    preds = out.argmax(dim=1)
    
    # Check if there is a predicted mask output
    print("\n--- Evaluation Start ---")

def evaluate_mask(mask, split_name):
    y_true = graph_data.y[mask].cpu().numpy()
    y_pred = preds[mask].cpu().numpy()
    y_prob = probs[mask][:, 1].cpu().numpy() # Probability of class 1 (fraud)
    
    acc = accuracy_score(y_true, y_pred)
    print(f"\n--- {split_name} Results ---")
    print(f"Accuracy:  {acc:.4f}")

    if split_name == 'Test':
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        # ROC AUC Requires probabilities of the positive class
        try:
            roc_auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            roc_auc = float('nan') # Handles edge case where test set has no positive instances
            
        cm = confusion_matrix(y_true, y_pred)
        
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        print(f"ROC-AUC:   {roc_auc:.4f}")
        print(f"Confusion Matrix:\n{cm}")

evaluate_mask(graph_data.train_mask, 'Train')
evaluate_mask(graph_data.val_mask, 'Validation')
evaluate_mask(graph_data.test_mask, 'Test')



--- Evaluation Start ---

--- Train Results ---
Accuracy:  0.0394

--- Validation Results ---
Accuracy:  0.0400

--- Test Results ---
Accuracy:  0.0387
Precision: 0.0387
Recall:    0.9667
F1-Score:  0.0745
ROC-AUC:   0.5833
Confusion Matrix:
[[  0 720]
 [  1  29]]


C:\Users\JHANANISHRI\AppData\Local\Temp\ipykernel_7568\409515191.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


### 8️⃣ Real-World Adaptation for Fraud Detection

Once this GraphSAGE model is trained and saved to `best_gnn_model.pt`, it can be deployed into a real-time production pipeline:
1. **Graph Construction:** When a new transaction occurs, it is processed into a node with its associated features $X_{new}$ based on transaction amount, IPs, devices, etc.
2. **Dynamic Edges:** Edges are instantly drawn between this new transaction node and existing nodes (e.g., shared IP, shared device, or shared accounts).
3. **Inference Flow:** The localized sub-graph representing the new node and its neighbors is passed through the loaded Pytorch Geometric model. GraphSAGE excels here because it is inductive; it generates embeddings by sampling and aggregating features from a node's local neighborhood, without needing to see that specific node during training.
4. **Actionable Alerts:** The model outputs a probability score for fraud. If the probability exceeds a predefined business threshold, the system flags the transaction for manual review, implements a 2FA challenge, or freezes the transaction.